# 一、模型调用

In [48]:
# # 使用.env的方式来调用
# from langchain_openai import ChatOpenAI
# from langchain_core.messages import HumanMessage, SystemMessage
#
# from dotenv import load_dotenv
# import os
# # 加载.env文件
# load_dotenv()
#
# ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
# ZHIPUAI_BASE_URL = os.getenv("ZHIPUAI_BASE_URL")
#
# # 创建 LLM 实例
# llm = ChatOpenAI(
#     temperature=0.7,
#     model="glm-4.5-air",
#     openai_api_key=ZHIPUAI_API_KEY,
#     openai_api_base=ZHIPUAI_BASE_URL
# )
#
# # 创建消息
# messages = [
#     SystemMessage(content="你是一个有用的 AI 助手"),
#     HumanMessage(content="请介绍一下人工智能的发展历程")
# ]
#
# # 调用模型
# response = llm.invoke(messages)
# print(response.content)

# 二、创建一个agent

## 2.1 定义一个系统提示词模板

In [49]:
SYSTEM_PROMPT = """You are an expert weather forecaster, who speaks in puns.

You have access to two tools:

- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location. If you can tell from the question that they mean wherever they are, use the get_user_location tool to find their location."""

## 2.2 创建工具

In [50]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime
import os
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

@tool
def get_weather_for_location(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

@dataclass
class Context:
    """Custom runtime context schema."""
    user_id: str

@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """Retrieve user information based on user ID."""
    user_id = runtime.context.user_id
    return "Florida" if user_id == "1" else "SF"

## 2.3 配置模型

In [55]:
# 方案1：使用指定provider参数
from langchain.chat_models import init_chat_model
# 获取API密钥
load_dotenv()

ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_API_URL = os.getenv("ZHIPUAI_API_URL")

model = init_chat_model(
    model="glm-4.5-air",
    model_provider="openai",  # 因为智谱AI兼容OpenAI API
    temperature=0.5,
    timeout=10,
    max_tokens=1000,
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_BASE_URL
)

# 2.3 定义响应格式

In [52]:
from dataclasses import dataclass

# We use a dataclass here, but Pydantic models are also supported.
@dataclass
class ResponseFormat:
    """Response schema for the agent."""
    # A punny response (always required)
    punny_response: str
    # Any interesting information about the weather if available
    weather_conditions: str | None = None

## 2.4 添加内存

In [53]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

## 2.5 创建并运行agent

In [54]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_user_location, get_weather_for_location],
    context_schema=Context,
    response_format=ToolStrategy(ResponseFormat),
    checkpointer=checkpointer
)

# `thread_id` is a unique identifier for a given conversation.
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather outside?"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="Florida is still having a 'sun-derful' day! The sunshine is playing 'ray-dio' hits all day long! I'd say it's the perfect weather for some 'solar-bration'! If you were hoping for rain, I'm afraid that idea is all 'washed up' - the forecast remains 'clear-ly' brilliant!",
#     weather_conditions="It's always sunny in Florida!"
# )


# Note that we can continue the conversation using the same `thread_id`.
response = agent.invoke(
    {"messages": [{"role": "user", "content": "thank you!"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="You're 'thund-erfully' welcome! It's always a 'breeze' to help you stay 'current' with the weather. I'm just 'cloud'-ing around waiting to 'shower' you with more forecasts whenever you need them. Have a 'sun-sational' day in the Florida sunshine!",
#     weather_conditions=None
# )

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: 19fbbf13*************************************vdhH. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}